# 05 — Notifications & Scheduling

This notebook sets up automated delivery of Claude's portfolio recommendations via:
- **Email** (SendGrid — free tier: 100 emails/day)
- **Slack** (optional — incoming webhook)
- **Pushover** (optional — mobile push notifications)

It also includes a **full pipeline runner** that chains notebooks 01→04 and delivers results,
plus a **scheduler** for daily/weekly automated runs.

**Prerequisites:**
- Run notebooks 01–04 first (so `reports/latest_analysis.json` exists)
- SendGrid API key (free at https://signup.sendgrid.com)
- Run `uv sync` to install sendgrid + schedule

## Setup

In [1]:
import json
import os
import sys
from datetime import datetime

from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"), override=True)

# Load latest analysis
REPORT_FILE = os.path.join("..", "reports", "latest_analysis.json")

if not os.path.exists(REPORT_FILE):
    raise FileNotFoundError(
        "latest_analysis.json not found. Run notebook 04 first."
    )

with open(REPORT_FILE, "r") as f:
    report = json.load(f)

analysis = report["analysis"]
print(f"✅ Loaded analysis from {report['generated_at']}")
print(f"   Model: {report['model']}")
print(f"   Recommendations: {len(analysis.get('recommendations', []))}")

✅ Loaded analysis from 2026-03-20T15:09:55.687963
   Model: claude-sonnet-4-6
   Recommendations: 14


## Build Email Report

We format Claude's analysis into a clean HTML email.

In [2]:
import os

# Brokerage badge colors — one distinct color per app
BROKERAGE_COLORS = {
    "Stash":     ("#1d4ed8", "#93c5fd"),   # blue
    "Robinhood": ("#065f46", "#6ee7b7"),   # green
    "Sofi":      ("#6b21a8", "#d8b4fe"),   # purple
    "Acorns":    ("#92400e", "#fcd34d"),   # amber
}

def build_email_html(analysis: dict, report_meta: dict) -> str:
    """Convert Claude's analysis JSON into a verbose HTML report.
    
    Each recommendation renders as a collapsible <details> card showing:
      bull_case, bear_case, key_signals, risk_factors, position_note
    Top-level sections: health, counts, sector concentration, action items,
    recommendations, watchlist, generated-by footer.
    Works in any browser and in Gmail (iPhone/Android).
    """

    pa   = analysis.get("portfolio_assessment", {})
    recs = analysis.get("recommendations", [])
    wl   = analysis.get("watchlist", [])
    acts = analysis.get("action_items", [])

    health_color = {
        "strong":   "#22c55e",
        "moderate": "#eab308",
        "weak":     "#ef4444",
    }.get(str(pa.get("overall_health", "")).lower(), "#888")

    buys  = sum(1 for r in recs if r.get("action") == "BUY")
    sells = sum(1 for r in recs if r.get("action") == "SELL")
    holds = sum(1 for r in recs if r.get("action") == "HOLD")

    # ── Recommendation cards ────────────────────────────────────────────────
    def pill(text, bg, fg="#fff"):
        return (f'<span style="display:inline-block;padding:2px 8px;border-radius:4px;'
                f'background:{bg};color:{fg};font-size:11px;font-weight:bold;">{text}</span>')

    def brokerage_badge(name):
        bg, fg = BROKERAGE_COLORS.get(name, ("#374151", "#d1d5db"))
        return (f'<span style="display:inline-block;padding:1px 7px;border-radius:10px;'
                f'background:{bg};color:{fg};font-size:10px;font-weight:600;'
                f'letter-spacing:0.03em;">{name}</span>')

    def signal_chips(signals):
        if not signals:
            return ""
        chips = "".join(
            f'<span style="display:inline-block;margin:2px;padding:2px 7px;'
            f'border-radius:12px;background:#1e3a5f;color:#93c5fd;font-size:11px;">'
            f'{s}</span>'
            for s in signals
        )
        return f'<div style="margin-top:4px;">{chips}</div>'

    action_cfg = {
        "BUY":  ("#064e3b", "#22c55e"),
        "SELL": ("#450a0a", "#ef4444"),
        "HOLD": ("#422006", "#eab308"),
    }

    rec_cards = ""
    for r in recs:
        action    = r.get("action", "HOLD")
        ticker    = r.get("ticker", "?")
        name      = r.get("name", "")
        brokerage = r.get("brokerage", "")
        conf      = r.get("confidence", "")
        urg       = r.get("urgency", "")
        thesis    = r.get("thesis", "")
        bull      = r.get("bull_case", "")
        bear      = r.get("bear_case", "")
        signals   = r.get("key_signals", [])
        risks     = r.get("risk_factors", [])
        note      = r.get("position_note", "")

        bg, ac = action_cfg.get(action, ("#333", "#888"))

        conf_color = {"high": "#22c55e", "medium": "#eab308", "low": "#ef4444"}.get(
            str(conf).lower(), "#aaa"
        )
        urg_color = {"soon": "#f97316", "monitor": "#eab308", "none": "#888"}.get(
            str(urg).lower(), "#aaa"
        )

        risks_html = ""
        if risks:
            items = "".join(f"<li>{rv}</li>" for rv in risks)
            risks_html = (
                f'<p style="margin:6px 0 2px;color:#f87171;font-size:12px;font-weight:bold;">⚠️ Risk Factors</p>'
                f'<ul style="margin:2px 0;padding-left:18px;color:#fca5a5;font-size:12px;">{items}</ul>'
            )

        extra_html = ""
        if bull:
            extra_html += (
                f'<p style="margin:6px 0 2px;color:#86efac;font-size:12px;font-weight:bold;">📈 Bull Case</p>'
                f'<p style="margin:0;color:#d1fae5;font-size:12px;">{bull}</p>'
            )
        if bear:
            extra_html += (
                f'<p style="margin:6px 0 2px;color:#fca5a5;font-size:12px;font-weight:bold;">📉 Bear Case</p>'
                f'<p style="margin:0;color:#fee2e2;font-size:12px;">{bear}</p>'
            )
        if signals:
            extra_html += (
                f'<p style="margin:6px 0 2px;color:#93c5fd;font-size:12px;font-weight:bold;">📡 Key Signals</p>'
                + signal_chips(signals)
            )
        if risks_html:
            extra_html += risks_html
        if note:
            extra_html += (
                f'<p style="margin:6px 0 2px;color:#fde68a;font-size:12px;font-weight:bold;">📝 Position Note</p>'
                f'<p style="margin:0;color:#fef3c7;font-size:12px;">{note}</p>'
            )

        has_detail = bool(bull or bear or signals or risks or note)
        detail_block = ""
        if has_detail:
            detail_block = (
                f'<details style="margin-top:8px;">'
                f'<summary style="cursor:pointer;color:#93c5fd;font-size:12px;'
                f'font-weight:bold;list-style:none;">▶ Details</summary>'
                f'<div style="margin-top:8px;padding:10px;background:#0f172a;'
                f'border-radius:6px;">{extra_html}</div>'
                f'</details>'
            )

        broker_badge = brokerage_badge(brokerage) if brokerage else ""

        rec_cards += f"""
        <div style="background:{bg};border-left:4px solid {ac};border-radius:6px;
                    padding:12px 14px;margin:8px 0;">
            <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
                {pill(action, ac, "#000" if action=="HOLD" else "#fff")}
                <strong style="color:#fff;font-size:15px;">{ticker}</strong>
                {broker_badge}
                <span style="color:#aaa;font-size:13px;">{name[:28]}</span>
                <span style="margin-left:auto;font-size:11px;">
                    <span style="color:{conf_color};">● {conf}</span>
                    &nbsp;|&nbsp;
                    <span style="color:{urg_color};">{urg}</span>
                </span>
            </div>
            <p style="margin:8px 0 0;color:#cbd5e1;font-size:13px;">{thesis}</p>
            {detail_block}
        </div>
        """

    # ── Sector concentration ────────────────────────────────────────────────
    sector_html = ""
    sector_text = pa.get("sector_concentration", "")
    if sector_text:
        sector_html = f"""
        <div style="background:#16213e;border-radius:8px;padding:16px;margin:16px 0;">
            <h2 style="margin-top:0;color:#fff;">🏭 Sector Concentration</h2>
            <p style="color:#cbd5e1;font-size:14px;line-height:1.6;">{sector_text}</p>
        </div>
        """

    # ── Action items ────────────────────────────────────────────────────────
    action_html = "".join(f'<li style="margin-bottom:6px;">{a}</li>' for a in acts)

    # ── Watchlist ───────────────────────────────────────────────────────────
    wl_html = "".join(
        f'<li style="margin-bottom:4px;"><strong>{w.get("ticker","?")}</strong>: {w.get("reason","")}</li>'
        for w in wl
    )
    watchlist_block = f"""
        <div style="background:#1a1a2e;border:1px solid #333;border-radius:8px;
                    padding:16px;margin:16px 0;">
            <h2 style="margin-top:0;color:#fff;">👀 Watchlist</h2>
            <ul style="padding-left:20px;">{wl_html}</ul>
        </div>
    """ if wl else ""

    # ── Footer ──────────────────────────────────────────────────────────────
    model       = report_meta.get("model", "claude")
    gen_at      = report_meta.get("generated_at", "")[:16]
    usage       = report_meta.get("usage", {})
    in_tok      = usage.get("input_tokens", 0)
    out_tok     = usage.get("output_tokens", 0)
    # Approximate cost: Sonnet 4.6 = $3/$15 per 1M tokens
    cost_usd    = (in_tok * 3 + out_tok * 15) / 1_000_000

    # ── Brokerage legend ────────────────────────────────────────────────────
    legend_items = "".join(
        f'<span style="display:inline-flex;align-items:center;gap:5px;margin:3px 6px;">'
        f'<span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{fg};"></span>'
        f'<span style="color:#aaa;font-size:12px;">{b}</span></span>'
        for b, (_, fg) in BROKERAGE_COLORS.items()
    )
    legend_html = f'<div style="margin:0 0 16px;padding:10px 14px;background:#16213e;border-radius:6px;">' \
                  f'<span style="color:#666;font-size:11px;margin-right:8px;">Brokerages:</span>{legend_items}</div>'

    footer_html = f"""
        <div style="text-align:center;padding:16px 0;border-top:1px solid #333;
                    color:#555;font-size:11px;margin-top:20px;">
            <p style="margin:4px 0;">⚠️ AI-generated analysis — not financial advice. Do your own research.</p>
            <p style="margin:4px 0;">
                Generated {gen_at} · {model} ·
                {in_tok:,} in / {out_tok:,} out tokens ·
                ≈ ${cost_usd:.3f}
            </p>
            <p style="margin:4px 0;">Financial Analyst Agent — Powered by Claude</p>
        </div>
    """

    html = f"""<!DOCTYPE html>
<html>
<head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"></head>
<body style="background:#1a1a2e;color:#e0e0e0;
             font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
             max-width:720px;margin:0 auto;padding:20px;">

    <!-- Header -->
    <div style="text-align:center;padding:20px 0;border-bottom:2px solid #333;">
        <h1 style="margin:0;color:#fff;">📊 Portfolio Analysis</h1>
        <p style="color:#888;margin:5px 0;">{gen_at} | {model}</p>
    </div>

    <!-- Health -->
    <div style="background:#16213e;border-radius:8px;padding:16px;margin:16px 0;">
        <h2 style="margin-top:0;color:#fff;">
            Portfolio Health:
            <span style="color:{health_color};">{str(pa.get("overall_health","N/A")).upper()}</span>
        </h2>
        <p style="color:#cbd5e1;">{pa.get("summary","")}</p>
        <p style="color:#f59e0b;">⚠️ Top Concern: {pa.get("top_concern","N/A")}</p>
    </div>

    <!-- Counts -->
    <div style="display:flex;gap:12px;margin:16px 0;text-align:center;">
        <div style="flex:1;background:#064e3b;border-radius:8px;padding:12px;">
            <div style="font-size:28px;font-weight:bold;color:#22c55e;">{buys}</div>
            <div style="color:#aaa;font-size:13px;">BUY</div>
        </div>
        <div style="flex:1;background:#422006;border-radius:8px;padding:12px;">
            <div style="font-size:28px;font-weight:bold;color:#eab308;">{holds}</div>
            <div style="color:#aaa;font-size:13px;">HOLD</div>
        </div>
        <div style="flex:1;background:#450a0a;border-radius:8px;padding:12px;">
            <div style="font-size:28px;font-weight:bold;color:#ef4444;">{sells}</div>
            <div style="color:#aaa;font-size:13px;">SELL</div>
        </div>
    </div>

    {sector_html}

    <!-- Action Items -->
    <div style="background:#16213e;border-radius:8px;padding:16px;margin:16px 0;">
        <h2 style="margin-top:0;color:#fff;">🎯 Action Items</h2>
        <ol style="padding-left:20px;color:#cbd5e1;">{action_html}</ol>
    </div>

    <!-- Recommendations -->
    <h2 style="color:#fff;">Recommendations
        <span style="font-size:13px;font-weight:normal;color:#888;">
            — click ▶ Details on any card to expand
        </span>
    </h2>
    {legend_html}
    {rec_cards}

    {watchlist_block}

    {footer_html}
</body>
</html>"""

    return html


# ── Build + save to disk ────────────────────────────────────────────────────
email_html = build_email_html(analysis, report)
print(f"✅ HTML report generated ({len(email_html):,} chars)")

# Save to reports/ directory (two copies: timestamped + latest)
reports_dir = os.path.join("..", "reports")
os.makedirs(reports_dir, exist_ok=True)

ts = datetime.now().strftime("%Y-%m-%d_%H%M")
timestamped_path = os.path.join(reports_dir, f"analysis_{ts}.html")
latest_path      = os.path.join(reports_dir, "latest_analysis.html")

with open(timestamped_path, "w", encoding="utf-8") as f:
    f.write(email_html)
with open(latest_path, "w", encoding="utf-8") as f:
    f.write(email_html)

print(f"💾 Saved → {timestamped_path}")
print(f"💾 Saved → {latest_path}  (overwritten)")


✅ HTML report generated (55,018 chars)
💾 Saved → ../reports/analysis_2026-03-20_1509.html
💾 Saved → ../reports/latest_analysis.html  (overwritten)


### Preview Email

In [3]:
from IPython.display import HTML, display

display(HTML(email_html))

## Send via Email (SendGrid)

### Setup:
1. Sign up at https://signup.sendgrid.com (free: 100 emails/day)
2. Create an API key at Settings → API Keys → Create API Key (Full Access)
3. Verify a Sender Identity (Settings → Sender Authentication → Single Sender Verification)
4. Add to your `.env`:
```
SENDGRID_API_KEY=SG.xxxxx
SENDGRID_FROM_EMAIL=your-verified-sender@email.com
NOTIFICATION_EMAIL=your-personal@email.com
```

In [4]:
# ── SendGrid email sending — DISABLED ───────────────────────────────────────
# Dashboard is now live on Vercel — email reports are no longer needed.
# To re-enable: change EMAIL_ENABLED to True below.
EMAIL_ENABLED = False

from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail, Email, To, Content

SENDGRID_API_KEY = os.getenv("SENDGRID_API_KEY")
FROM_EMAIL = os.getenv("SENDGRID_FROM_EMAIL")
TO_EMAIL = os.getenv("NOTIFICATION_EMAIL")

if not EMAIL_ENABLED:
    print("⬜ Email (SendGrid) disabled — using Vercel dashboard instead.")
    print("   To re-enable: set EMAIL_ENABLED = True in this cell.")
elif not all([SENDGRID_API_KEY, FROM_EMAIL, TO_EMAIL]):
    print("⚠️  SendGrid not configured. Add these to .env:")
    print("   SENDGRID_API_KEY=SG.xxxxx")
    print("   SENDGRID_FROM_EMAIL=verified@email.com")
    print("   NOTIFICATION_EMAIL=you@email.com")
    print("\n   Skipping email send. You can still use Slack or Pushover below.")
else:
    # Build email
    pa = analysis.get("portfolio_assessment", {})
    recs = analysis.get("recommendations", [])
    buys = sum(1 for r in recs if r.get("action") == "BUY")
    sells = sum(1 for r in recs if r.get("action") == "SELL")
    holds = sum(1 for r in recs if r.get("action") == "HOLD")

    date_str = datetime.now().strftime("%b %d")
    subject = f"📊 Portfolio Report ({date_str}): {buys} Buy, {holds} Hold, {sells} Sell — {pa.get('overall_health', 'N/A').title()}"

    message = Mail(
        from_email=Email(FROM_EMAIL),
        to_emails=To(TO_EMAIL),
        subject=subject,
        html_content=Content("text/html", email_html),
    )

    try:
        sg = SendGridAPIClient(SENDGRID_API_KEY)
        response = sg.send(message)
        print(f"✅ Email sent! Status: {response.status_code}")
        print(f"   To: {TO_EMAIL}")
        print(f"   Subject: {subject}")
    except Exception as e:
        print(f"❌ Email failed: {e}")

⬜ Email (SendGrid) disabled — using Vercel dashboard instead.
   To re-enable: set EMAIL_ENABLED = True in this cell.


## Send via Slack (Optional)

### Setup:
1. Go to https://api.slack.com/apps → Create New App → From Scratch
2. Enable Incoming Webhooks → Add New Webhook → Select a channel
3. Copy the webhook URL to `.env`:
```
SLACK_WEBHOOK_URL=https://hooks.slack.com/services/T.../B.../xxx
```

In [5]:
# import requests
# 
# SLACK_WEBHOOK = os.getenv("SLACK_WEBHOOK_URL")
# 
# def build_slack_message(analysis: dict) -> dict:
#     """Build a Slack Block Kit message from analysis."""
#     pa = analysis.get("portfolio_assessment", {})
#     recs = analysis.get("recommendations", [])
#     actions = analysis.get("action_items", [])
#     
#     health_emoji = {"strong": ":large_green_circle:", "moderate": ":yellow_circle:", "weak": ":red_circle:"}.get(pa.get("overall_health"), ":white_circle:")
#     
#     # Build recommendation lines
#     rec_lines = []
#     for r in recs:
#         emoji = {"BUY": ":chart_with_upwards_trend:", "SELL": ":chart_with_downwards_trend:", "HOLD": ":pause_button:"}.get(r.get("action"), "")
#         rec_lines.append(f"{emoji} *{r.get('action')}* `{r.get('ticker')}` — {r.get('thesis', '')[:80]}")
#     
#     rec_text = "\n".join(rec_lines)
#     
#     action_text = "\n".join(f"{i}. {a}" for i, a in enumerate(actions, 1))
#     
#     return {
#         "blocks": [
#             {"type": "header", "text": {"type": "plain_text", "text": "📊 Portfolio Analysis Report"}},
#             {"type": "section", "text": {"type": "mrkdwn", "text": f"{health_emoji} *Portfolio Health: {pa.get('overall_health', 'N/A').upper()}*\n{pa.get('summary', '')}"}},
#             {"type": "divider"},
#             {"type": "section", "text": {"type": "mrkdwn", "text": f"*Recommendations:*\n{rec_text}"}},
#             {"type": "divider"},
#             {"type": "section", "text": {"type": "mrkdwn", "text": f"*:dart: Action Items:*\n{action_text}"}},
#             {"type": "context", "elements": [{"type": "mrkdwn", "text": "_AI-generated analysis. Not financial advice._"}]},
#         ]
#     }
# 
# if not SLACK_WEBHOOK:
#     print("⚠️  Slack not configured. Add SLACK_WEBHOOK_URL to .env to enable.")
#     print("   Skipping Slack notification.")
# else:
#     slack_msg = build_slack_message(analysis)
#     response = requests.post(SLACK_WEBHOOK, json=slack_msg)
#     
#     if response.status_code == 200:
#         print("✅ Slack message sent!")
#     else:
#         print(f"❌ Slack failed: {response.status_code} — {response.text}")

print("⬜ Slack notifications disabled (not configured)")

⬜ Slack notifications disabled (not configured)


## Send via Pushover (Mobile Push Notification)

Sends a compact alert to your phone the moment the pipeline finishes — before you've even
opened email. Keys are loaded from `.env`.

**Message format:**
```
Title:   📊 Portfolio: MODERATE
Message: 6 Buy · 45 Hold · 15 Sell
         ⚠️ [top_concern — first 100 chars]
```


In [6]:
import requests

PUSHOVER_USER  = os.getenv("PUSHOVER_USER_KEY")
PUSHOVER_TOKEN = os.getenv("PUSHOVER_APP_TOKEN")

if not all([PUSHOVER_USER, PUSHOVER_TOKEN]):
    print("⬜ Pushover not configured — add PUSHOVER_USER_KEY and PUSHOVER_APP_TOKEN to .env.")
else:
    pa      = analysis.get("portfolio_assessment", {})
    recs    = analysis.get("recommendations", [])
    buys    = sum(1 for r in recs if r.get("action") == "BUY")
    sells   = sum(1 for r in recs if r.get("action") == "SELL")
    holds   = sum(1 for r in recs if r.get("action") == "HOLD")
    health  = str(pa.get("overall_health", "?")).upper()
    concern = str(pa.get("top_concern", ""))[:100]

    title = f"📊 Portfolio: {health}"
    msg   = f"{buys} Buy · {holds} Hold · {sells} Sell\n⚠️ {concern}"

    resp = requests.post(
        "https://api.pushover.net/1/messages.json",
        data={
            "token":    PUSHOVER_TOKEN,
            "user":     PUSHOVER_USER,
            "title":    title,
            "message":  msg,
            "priority": 0,
        },
        timeout=15,
    )

    if resp.status_code == 200:
        print(f"✅ Pushover sent: {title}")
        print(f"   {buys} Buy · {holds} Hold · {sells} Sell")
    else:
        print(f"❌ Pushover failed ({resp.status_code}): {resp.text}")


⬜ Pushover not configured — add PUSHOVER_USER_KEY and PUSHOVER_APP_TOKEN to .env.


## Full Pipeline Runner

This cell runs the entire pipeline end-to-end: refresh holdings → enrich → analyze → notify.
You can call this from a cron job or scheduler.

In [7]:
# Save the pipeline as a standalone script

pipeline_script = '''#!/usr/bin/env python3
"""Full pipeline: refresh → enrich → analyze → notify.

Run manually:   uv run python run_pipeline.py
Run via cron:   0 7 * * 1-5 cd /path/to/financial-agent && uv run python run_pipeline.py
"""

import json
import os
import sys
import subprocess
from datetime import datetime

PROJECT_DIR = os.path.dirname(os.path.abspath(__file__))
NOTEBOOKS_DIR = os.path.join(PROJECT_DIR, "notebooks")


def run_notebook(notebook_name: str) -> bool:
    """Execute a notebook via jupyter and return success status."""
    path = os.path.join(NOTEBOOKS_DIR, notebook_name)
    print(f"\\n{'='*60}")
    print(f"Running {notebook_name}...")
    print(f"{'='*60}")
    
    result = subprocess.run(
        [
            sys.executable, "-m", "jupyter", "nbconvert",
            "--to", "notebook",
            "--execute",
            "--ExecutePreprocessor.timeout=300",
            "--output", notebook_name,
            path,
        ],
        capture_output=True,
        text=True,
        cwd=NOTEBOOKS_DIR,
    )
    
    if result.returncode != 0:
        print(f"FAILED: {notebook_name}")
        print(result.stderr[-500:] if result.stderr else "No error output")
        return False
    
    print(f"✅ {notebook_name} complete")
    return True


def main():
    start = datetime.now()
    print(f"🚀 Pipeline started at {start.strftime('%Y-%m-%d %H:%M:%S')}")
    
    notebooks = [
        # Skip 01 — tokens are already saved and persistent
        "02_fetch_holdings.ipynb",
        "03_data_enrichment.ipynb",
        "04_claude_analysis.ipynb",
        "05_notifications.ipynb",
    ]
    
    for nb in notebooks:
        if not run_notebook(nb):
            print(f"\\n❌ Pipeline failed at {nb}")
            sys.exit(1)
    
    elapsed = (datetime.now() - start).total_seconds()
    print(f"\\n🎉 Pipeline complete in {elapsed:.0f}s")


if __name__ == "__main__":
    main()
'''

pipeline_path = os.path.join("..", "run_pipeline.py")
with open(pipeline_path, "w") as f:
    f.write(pipeline_script)

print(f"💾 Pipeline script saved to {pipeline_path}")
print(f"")
print(f"   Manual run:   uv run python run_pipeline.py")
print(f"   Cron (weekdays 7am):")
print(f"   0 7 * * 1-5 cd /path/to/financial-agent && uv run python run_pipeline.py >> pipeline.log 2>&1")

💾 Pipeline script saved to ../run_pipeline.py

   Manual run:   uv run python run_pipeline.py
   Cron (weekdays 7am):
   0 7 * * 1-5 cd /path/to/financial-agent && uv run python run_pipeline.py >> pipeline.log 2>&1


## Scheduling (Interactive)

For quick testing, you can run the pipeline on a schedule right here in the notebook.
For production, use the cron approach above.

In [8]:
import schedule
import time
import subprocess

def run_full_pipeline():
    """Run the full pipeline script."""
    print(f"\n⏰ Scheduled run at {datetime.now().strftime('%H:%M:%S')}")
    subprocess.run([sys.executable, os.path.join("..", "run_pipeline.py")], cwd=os.path.join(".."))

# Uncomment to schedule:
# schedule.every().day.at("07:00").do(run_full_pipeline)   # Daily at 7 AM
# schedule.every().monday.at("07:00").do(run_full_pipeline)  # Weekly on Monday

# Uncomment to start the scheduler (runs indefinitely — interrupt kernel to stop):
# print("📅 Scheduler running. Press Ctrl+C or interrupt kernel to stop.")
# while True:
#     schedule.run_pending()
#     time.sleep(60)

print("📅 Scheduler ready. Uncomment the lines above to activate.")
print("   For production, use cron with run_pipeline.py instead.")

📅 Scheduler ready. Uncomment the lines above to activate.
   For production, use cron with run_pipeline.py instead.


## Notification Status Summary

In [9]:
print("═" * 52)
print("      NOTIFICATION CHANNELS")
print("═" * 52)
print(f"  📧 Email (SendGrid):   {'✅ Configured' if SENDGRID_API_KEY else '⬜ Not configured'}")
print(f"  📱 Pushover:           {'✅ Configured' if (PUSHOVER_USER and PUSHOVER_TOKEN) else '⬜ Not configured'}")
print(f"  💬 Slack:              ⬜ Disabled")
print(f"  🔄 Pipeline Script:    ✅ Saved to run_pipeline.py")
print(f"  ⏰ launchd Schedule:   ✅ Tue 8am + Thu 8am")
print("═" * 52)
print()
print(f"  📄 HTML Report:")
print(f"     reports/latest_analysis.html  (browser-ready)")
print(f"     reports/analysis_{ts}.html")
print()
print(f"  💡 Email body = full verbose report (reads on iPhone Gmail)")
print(f"     Pushover  = quick ping: health + counts")


════════════════════════════════════════════════════
      NOTIFICATION CHANNELS
════════════════════════════════════════════════════
  📧 Email (SendGrid):   ⬜ Not configured
  📱 Pushover:           ⬜ Not configured
  💬 Slack:              ⬜ Disabled
  🔄 Pipeline Script:    ✅ Saved to run_pipeline.py
  ⏰ launchd Schedule:   ✅ Tue 8am + Thu 8am
════════════════════════════════════════════════════

  📄 HTML Report:
     reports/latest_analysis.html  (browser-ready)
     reports/analysis_2026-03-20_1509.html

  💡 Email body = full verbose report (reads on iPhone Gmail)
     Pushover  = quick ping: health + counts
